# EfficientNetV2 — Combined ASD + Emotion Detection
**Split: 80% Train / 10% Validation / 10% Test**

## Two Separate Datasets Required
**DATASET 1:**
```
asd_dataset.zip
├── ASD/
└── Non_ASD/
```

**DATASET 2 (ASD children face reactions):**
```
emotion_dataset.zip
├── anger/
├── joy/
├── sadness/
└── surprise/
```

## Outputs
`efficientnetv2_asd_model.h5` `efficientnetv2_emotion_model.h5` `efficientnetv2_asd_metrics.json` `efficientnetv2_emotion_metrics.json`

> **Runtime → GPU (T4 or A100)**

In [ ]:
# MediaPipe removed — mp.solutions unavailable in Colab with TF 2.19
# Using OpenCV proportional ROI masks for face region XAI instead.
print("XAI: using OpenCV proportional face-region masks (no mediapipe needed)")


In [ ]:
IMG_SIZE   = 224
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 1e-3
XAI_LAYER  = "top_activation"
MODEL_NAME = "efficientnetv2"
ASD_OUT    = "efficientnetv2_asd_model.h5"
EMO_OUT    = "efficientnetv2_emotion_model.h5"
TRAIN_R = 0.80; VAL_R = 0.10  # test = remaining 10%
# ASD: ASD=0, Non_ASD=1 (alphabetical); sigmoid > 0.5 → Non_ASD
ASD_CLASSES = ["ASD", "Non_ASD"]
EMO_CLASSES = ["anger", "joy", "sadness", "surprise"]
print("Config OK — 80/10/10 split")


## STEP 1: Upload ASD Dataset ZIP

In [ ]:
def find_or_extract(tag, required_sub, extract_to):
    # 1. Already extracted?
    for c in ["/content", extract_to, "/content/"+tag]:
        if os.path.isdir(os.path.join(c, required_sub)):
            print(f"Found {tag} at: {c}"); return c
    all_zips = glob.glob("/content/*.zip")
    if not all_zips:
        raise FileNotFoundError("No ZIP found — upload " + tag + " dataset ZIP to /content/ first!")
    # 2. Prefer a zip whose filename contains the tag (e.g. 'asd', 'emotion')
    tag_zips = [z for z in all_zips if tag.lower() in os.path.basename(z).lower()]
    # Also accept common synonyms
    if not tag_zips and tag == "asd":
        tag_zips = [z for z in all_zips if any(k in os.path.basename(z).lower()
                    for k in ["autism","autistic","asd"])]
    if not tag_zips and tag == "emotion":
        tag_zips = [z for z in all_zips if any(k in os.path.basename(z).lower()
                    for k in ["emotion","emo","expression","facial"])]
    # 3. Fall back to all ZIPs sorted by mtime (most recently uploaded first)
    candidates = tag_zips if tag_zips else sorted(all_zips, key=os.path.getmtime, reverse=True)
    last_err = None
    for z in candidates:
        dst = extract_to + "_" + os.path.splitext(os.path.basename(z))[0]
        print(f"Trying {os.path.basename(z)} -> {dst}")
        os.makedirs(dst, exist_ok=True)
        os.system(f'unzip -q -o "{z}" -d {dst}/')
        import time; time.sleep(1); os.system("sync")
        for root, dirs, _ in os.walk(dst):
            dirs[:] = [d for d in dirs if d not in [".ipynb_checkpoints","__MACOSX"]]
            if required_sub in dirs:
                print(f"  Found {required_sub}/ in {root}"); return root
        last_err = f"{required_sub}/ not found in {os.path.basename(z)}"
        print(f"  Skipping: {last_err}")
    raise FileNotFoundError(
        f"Could not find {required_sub}/ in any ZIP.\n"
        f"Make sure your {tag} ZIP contains a folder called '{required_sub}'.\n"
        f"Tried: {[os.path.basename(z) for z in candidates]}"
    )

def normalise_asd(root):
    for old,new in [("Autistic","ASD"),("Non_Autistic","Non_ASD"),("autism","ASD")]:
        p=os.path.join(root,old)
        if os.path.isdir(p) and not os.path.isdir(os.path.join(root,new)):
            os.rename(p,os.path.join(root,new)); print(f"Renamed {old}->{new}")
    for j in [".ipynb_checkpoints",".DS_Store","__MACOSX"]:
        jp=os.path.join(root,j)
        if os.path.exists(jp): shutil.rmtree(jp)

ASD_ROOT = find_or_extract("asd", "ASD", "/content/asd_raw")
normalise_asd(ASD_ROOT)
for cls in sorted(os.listdir(ASD_ROOT)):
    p=os.path.join(ASD_ROOT,cls)
    if os.path.isdir(p):
        n=len([f for f in os.listdir(p) if f.lower().endswith((".jpg",".jpeg",".png"))])
        print(f"  {cls}: {n} images")


## STEP 2: Upload Emotion Dataset ZIP
Delete the ASD zip from /content/ first, then upload emotion zip.
Folders: `anger/` `joy/` `sadness/` `surprise/`

In [ ]:
EMO_ROOT = find_or_extract("emotion", "anger", "/content/emo_raw")
for j in [".ipynb_checkpoints",".DS_Store","__MACOSX"]:
    jp=os.path.join(EMO_ROOT,j)
    if os.path.exists(jp): shutil.rmtree(jp)
emo_cls=sorted(d for d in os.listdir(EMO_ROOT) if os.path.isdir(os.path.join(EMO_ROOT,d)))
print(f"Emotion classes: {emo_cls}")
for cls in emo_cls:
    p=os.path.join(EMO_ROOT,cls)
    n=len([f for f in os.listdir(p) if f.lower().endswith((".jpg",".jpeg",".png"))])
    print(f"  {cls}: {n} images")


## STEP 3: Face-Crop (OpenCV Haar Cascade)

## STEP 4: 80 / 10 / 10 Split
| Folder | % | Purpose |
|--|--|--|
| `train/` | 80% | Training |
| `val/` | 10% | EarlyStopping |
| `test/` | 10% | Final unbiased metrics |

In [ ]:
import random, pathlib

def split_dataset(src_dir, dst_dir, train_r=0.80, val_r=0.10, seed=42):
    done_flag = os.path.join(dst_dir, ".split_done")
    if os.path.exists(done_flag):
        print(f"Using cached split: {dst_dir}"); return dst_dir
    random.seed(seed)
    exts = {".jpg",".jpeg",".png"}
    for cls in os.listdir(src_dir):
        cs = os.path.join(src_dir, cls)
        if not os.path.isdir(cs): continue
        files = [f for f in os.listdir(cs) if os.path.splitext(f)[1].lower() in exts]
        random.shuffle(files)
        n    = len(files); n_tr = int(n*train_r); n_vl = int(n*val_r)
        splits = {"train":files[:n_tr], "val":files[n_tr:n_tr+n_vl], "test":files[n_tr+n_vl:]}
        for split, flist in splits.items():
            out = os.path.join(dst_dir, split, cls); os.makedirs(out, exist_ok=True)
            for fn in flist: shutil.copy2(os.path.join(cs,fn), os.path.join(out,fn))
        print(f"  {cls}: {int(n*train_r)} train | {int(n*val_r)} val | {n-int(n*train_r)-int(n*val_r)} test")
    pathlib.Path(done_flag).touch()
    print(f"Split -> {dst_dir}"); return dst_dir

ASD_SPLIT=split_dataset(ASD_CROP,"/content/asd_split",TRAIN_R,VAL_R)
EMO_SPLIT=split_dataset(EMO_CROP,"/content/emo_split",TRAIN_R,VAL_R)
for sp in ["train","val","test"]:
    p=os.path.join(ASD_SPLIT,sp)
    t=sum(len([f for f in os.listdir(os.path.join(p,c)) if f.lower().endswith((".jpg",".jpeg",".png"))])
          for c in os.listdir(p) if os.path.isdir(os.path.join(p,c)))
    print(f"ASD  {sp}: {t}")
for sp in ["train","val","test"]:
    p=os.path.join(EMO_SPLIT,sp)
    t=sum(len([f for f in os.listdir(os.path.join(p,c)) if f.lower().endswith((".jpg",".jpeg",".png"))])
          for c in os.listdir(p) if os.path.isdir(os.path.join(p,c)))
    print(f"Emo  {sp}: {t}")


## STEP 5: Data Generators

In [ ]:
train_aug=ImageDataGenerator(rescale=1./255,rotation_range=20,width_shift_range=0.15,
    height_shift_range=0.15,horizontal_flip=True,zoom_range=0.15,
    brightness_range=[0.8,1.2],fill_mode="nearest")
val_aug=ImageDataGenerator(rescale=1./255)

asd_tr=train_aug.flow_from_directory(os.path.join(ASD_SPLIT,"train"),target_size=(224,224),batch_size=BATCH_SIZE,class_mode="binary",seed=42)
asd_vl=val_aug.flow_from_directory(os.path.join(ASD_SPLIT,"val"),  target_size=(224,224),batch_size=BATCH_SIZE,class_mode="binary",shuffle=False)
asd_te=val_aug.flow_from_directory(os.path.join(ASD_SPLIT,"test"), target_size=(224,224),batch_size=BATCH_SIZE,class_mode="binary",shuffle=False)
assert asd_tr.class_indices.get("ASD")==0, f"ERROR ASD must=0, got {asd_tr.class_indices}"
print(f"ASD  tr={asd_tr.n} vl={asd_vl.n} te={asd_te.n} | {asd_tr.class_indices}")

emo_tr=train_aug.flow_from_directory(os.path.join(EMO_SPLIT,"train"),target_size=(224,224),batch_size=BATCH_SIZE,class_mode="categorical",seed=42)
emo_vl=val_aug.flow_from_directory(os.path.join(EMO_SPLIT,"val"),  target_size=(224,224),batch_size=BATCH_SIZE,class_mode="categorical",shuffle=False)
emo_te=val_aug.flow_from_directory(os.path.join(EMO_SPLIT,"test"), target_size=(224,224),batch_size=BATCH_SIZE,class_mode="categorical",shuffle=False)
NC=len(emo_tr.class_indices)
print(f"Emo  tr={emo_tr.n} vl={emo_vl.n} te={emo_te.n} NC={NC} | {emo_tr.class_indices}")


## STEP 6: Train ASD Detection Model

In [ ]:
cbs=[EarlyStopping(patience=7,restore_best_weights=True,monitor="val_accuracy"),
     ReduceLROnPlateau(factor=0.3,patience=3,min_lr=1e-7)]
base_asd=EfficientNetV2B2(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_asd.trainable=False
x=layers.GlobalAveragePooling2D()(base_asd.output)
x=layers.Dense(256,activation="relu",name="asd_dense1")(x)
x=layers.BatchNormalization(name="asd_bn1")(x)
x=layers.Dropout(0.5)(x)
x=layers.Dense(64,activation="relu",name="asd_dense2")(x)
x=layers.Dropout(0.3)(x)
# sigmoid > 0.5 = Non_ASD (class 1)  |  <= 0.5 = ASD (class 0)
out=layers.Dense(1,activation="sigmoid",name="asd_output")(x)
asd_model=keras.Model(base_asd.input,out,name="asd_model")
asd_model.compile(optimizer=keras.optimizers.Adam(LR),loss="binary_crossentropy",metrics=["accuracy"])
print("--- Phase 1: frozen backbone ---")
asd_model.fit(asd_tr,epochs=EPOCHS,validation_data=asd_vl,callbacks=cbs)
base_asd.trainable=True
for l in base_asd.layers[:-30]: l.trainable=False
asd_model.compile(optimizer=keras.optimizers.Adam(LR/10),loss="binary_crossentropy",metrics=["accuracy"])
print("--- Phase 2: fine-tune ---")
asd_model.fit(asd_tr,epochs=20,validation_data=asd_vl,callbacks=cbs)
asd_model.save(ASD_OUT); print(f"Saved: {ASD_OUT}")


## STEP 7: Train Emotion Model (anger / joy / sadness / surprise)

In [ ]:
base_emo=EfficientNetV2B2(weights="imagenet",include_top=False,input_shape=(224,224,3))
base_emo.trainable=False
x=layers.GlobalAveragePooling2D()(base_emo.output)
x=layers.Dense(512,activation="relu",name="emo_dense1")(x)
x=layers.BatchNormalization(name="emo_bn1")(x)
x=layers.Dropout(0.5)(x)
x=layers.Dense(128,activation="relu",name="emo_dense2")(x)
x=layers.Dropout(0.3)(x)
out=layers.Dense(NC,activation="softmax",name="emo_output")(x)
emo_model=keras.Model(base_emo.input,out,name="emo_model")
emo_model.compile(optimizer=keras.optimizers.Adam(LR),loss="categorical_crossentropy",metrics=["accuracy"])
print(f"--- Phase 1: frozen (NC={NC}) ---")
emo_model.fit(emo_tr,epochs=EPOCHS,validation_data=emo_vl,callbacks=cbs)
base_emo.trainable=True
for l in base_emo.layers[:-30]: l.trainable=False
emo_model.compile(optimizer=keras.optimizers.Adam(LR/10),loss="categorical_crossentropy",metrics=["accuracy"])
print("--- Phase 2: fine-tune ---")
emo_model.fit(emo_tr,epochs=20,validation_data=emo_vl,callbacks=cbs)
emo_model.save(EMO_OUT); print(f"Saved: {EMO_OUT}")


## STEP 8: Final Metrics on TEST SET (10% unseen)
These are the honest numbers for your FYP report.

In [ ]:
def save_metrics(model, gen, title, fname, is_binary=False):
    print(f"\nTest evaluation: {title}")
    loss,acc=model.evaluate(gen,verbose=0)
    print(f"  Loss={loss:.4f}  Accuracy={acc*100:.1f}%")
    y_raw=model.predict(gen,verbose=0)
    y_pred=(y_raw.flatten()>0.5).astype(int) if is_binary else np.argmax(y_raw,axis=1)
    names=list(gen.class_indices.keys()); labels=list(range(len(names)))
    cm=confusion_matrix(gen.classes,y_pred,labels=labels)
    rep=classification_report(gen.classes,y_pred,target_names=names,labels=labels,output_dict=True)
    with open(fname,"w") as f: json.dump({"title":title,"test_accuracy":round(acc,4),
        "confusion_matrix":cm.tolist(),"classification_report":rep,"labels":names},f,indent=2)
    plt.figure(figsize=(max(4,len(names)+2),max(3,len(names)+1)))
    sns.heatmap(cm,annot=True,fmt="d",cmap="Blues",xticklabels=names,yticklabels=names)
    plt.title(f"Confusion Matrix (TEST SET) — {title}"); plt.tight_layout(); plt.show()
    print(classification_report(gen.classes,y_pred,target_names=names))
    print(f"Saved -> {fname}")

save_metrics(asd_model,asd_te,"EfficientNetV2 ASD","efficientnetv2_asd_metrics.json",is_binary=True)
save_metrics(emo_model,emo_te,"EfficientNetV2 Emotion","efficientnetv2_emotion_metrics.json")


## STEP 9: Clinical XAI — Grad-CAM + Facial Landmark Analysis
**Fix:** Robust Conv layer detection (works in Keras 2 & 3).

In [ ]:
# ── Guard: recreate generators if not defined (e.g. after runtime restart) ──
import pathlib as _pl
def _need(name):
    return name not in dir() or eval(name) is None if name in dir() else True

if "asd_te" not in dir() or "emo_te" not in dir() or "emo_tr" not in dir():
    print("Recreating generators (asd_te / emo_te)...")
    _val_aug = __import__("tensorflow").keras.preprocessing.image.ImageDataGenerator(rescale=1./255)
    _asd_sp  = "/content/asd_split"
    _emo_sp  = "/content/emo_split"
    if not os.path.isdir(os.path.join(_asd_sp,"test")):
        raise RuntimeError("Run STEP 4 first to create the 80/10/10 split!")
    import tensorflow as tf; from tensorflow.keras.preprocessing.image import ImageDataGenerator
    _aug2 = ImageDataGenerator(rescale=1./255)
    IMG_SIZE = IMG_SIZE if "IMG_SIZE" in dir() else 224
    BATCH_SIZE = BATCH_SIZE if "BATCH_SIZE" in dir() else 32
    asd_te = _aug2.flow_from_directory(os.path.join(_asd_sp,"test"),
        target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode="binary", shuffle=False)
    emo_tr = _aug2.flow_from_directory(os.path.join(_emo_sp,"train"),
        target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode="categorical", shuffle=False)
    emo_te = _aug2.flow_from_directory(os.path.join(_emo_sp,"test"),
        target_size=(IMG_SIZE,IMG_SIZE), batch_size=BATCH_SIZE,
        class_mode="categorical", shuffle=False)
    print(f"  asd_te={asd_te.n} emo_te={emo_te.n}")

def _find_layer(model, name):
    """Recursively search model (including sub-models) for a layer by name."""
    for l in model.layers:
        if l.name == name: return l
        if hasattr(l, "layers"):
            r = _find_layer(l, name)
            if r is not None: return r
    return None

def _find_last_conv(model):
    """
    Find the last Conv2D/DepthwiseConv2D/SeparableConv2D in the model.
    Uses isinstance — works in Keras 2 and 3 (output_shape is unreliable in Keras 3).
    """
    CONV_TYPES = (
        tf.keras.layers.Conv2D,
        tf.keras.layers.DepthwiseConv2D,
        tf.keras.layers.SeparableConv2D,
    )
    # Collect all layers, flatten sub-models
    def all_layers(m):
        out = []
        for l in m.layers:
            if hasattr(l, "layers"): out.extend(all_layers(l))
            else: out.append(l)
        return out

    for l in reversed(all_layers(model)):
        if isinstance(l, CONV_TYPES): return l
    return None

def get_gradcam(model, img_array, layer_name=XAI_LAYER):
    """Grad-CAM. Falls back gracefully when named layer is not found."""
    t = _find_layer(model, layer_name)
    if t is None:
        print(f"[Grad-CAM] Layer '{layer_name}' not found — using last Conv2D")
        t = _find_last_conv(model)
    if t is None:
        print("[Grad-CAM] WARNING: no Conv layer found — returning blank heatmap")
        h, w = img_array.shape[1], img_array.shape[2]
        return np.zeros((h, w), dtype=np.float32)

    try:
        gm = tf.keras.Model(model.inputs, [t.output, model.output])
        with tf.GradientTape() as tape:
            co, preds = gm(img_array, training=False)
            loss = preds[:, tf.argmax(preds[0])] if preds.shape[-1] > 1 else preds[:, 0]
        grads = tape.gradient(loss, co)
        if grads is None:
            print("[Grad-CAM] Gradients are None — returning blank heatmap")
            h, w = img_array.shape[1], img_array.shape[2]
            return np.zeros((h, w), dtype=np.float32)
        w_vec = tf.reduce_mean(grads, axis=(0, 1, 2))
        cam   = tf.reduce_sum(tf.multiply(co[0], w_vec), axis=-1).numpy()
        cam   = np.maximum(cam, 0)
        cam   = cv2.resize(cam, (img_array.shape[2], img_array.shape[1]))
        denom = cam.max()
        return cam / (denom + 1e-10) if denom > 0 else cam
    except Exception as e:
        print(f"[Grad-CAM] Error: {e} — returning blank heatmap")
        h, w = img_array.shape[1], img_array.shape[2]
        return np.zeros((h, w), dtype=np.float32)

# Proportional ROI masks based on normalised face coordinates.
# Works on any face image without MediaPipe.
# Each entry: (y0_frac, y1_frac, x0_frac, x1_frac)
PROP_ROIS = {
    "Forehead":    (0.00, 0.25, 0.15, 0.85),
    "Left Eye":    (0.22, 0.42, 0.05, 0.45),
    "Right Eye":   (0.22, 0.42, 0.55, 0.95),
    "Nose":        (0.38, 0.65, 0.30, 0.70),
    "Mouth/Teeth": (0.60, 0.82, 0.20, 0.80),
    "Chin":        (0.80, 1.00, 0.20, 0.80),
    "Left Ear":    (0.22, 0.65, 0.00, 0.12),
    "Right Ear":   (0.22, 0.65, 0.88, 1.00),
}
CMAPS = [plt.cm.spring, plt.cm.hot, plt.cm.hot, plt.cm.cool,
         plt.cm.autumn, plt.cm.summer, plt.cm.winter, plt.cm.winter]

def make_prop_masks(h, w):
    """Build binary masks from proportional ROIs for a face of size (h, w)."""
    masks = {}
    for name, (y0,y1,x0,x1) in PROP_ROIS.items():
        m = np.zeros((h, w), dtype=np.uint8)
        r0,r1 = int(y0*h), int(y1*h)
        c0,c1 = int(x0*w), int(x1*w)
        m[r0:r1, c0:c1] = 255
        masks[name] = m
    return masks

def sc(cam, mask):
    m = mask.astype(float)/255
    return round(float((cam*m).mean() / (cam.mean()+1e-8)), 2)

def clinical_xai(asd_m, emo_m, img_batch, img_disp, asd_label, emo_label):
    is_asd  = "ASD" in asd_label and "NON" not in asd_label.upper()
    h, w    = img_disp.shape[:2]
    asd_cam = get_gradcam(asd_m, img_batch)
    emo_cam = get_gradcam(emo_m, img_batch)
    masks   = make_prop_masks(h, w)   # always works — no mediapipe

    fig = plt.figure(figsize=(28, 13))
    col = "#c62828" if is_asd else "#1b5e20"
    fig.suptitle(f"ASD: {asd_label}   |   Emotion: {emo_label}",
                 fontsize=15, fontweight="bold", color=col)
    axes = [fig.add_subplot(3, 5, i+1) for i in range(15)]

    axes[0].imshow(img_disp); axes[0].set_title("Original", fontweight="bold"); axes[0].axis("off")
    axes[1].imshow(img_disp); axes[1].imshow(plt.cm.jet(asd_cam)[:,:,:3], alpha=0.45)
    axes[1].set_title("ASD Grad-CAM", fontweight="bold"); axes[1].axis("off")
    axes[2].imshow(img_disp); axes[2].imshow(plt.cm.jet(emo_cam)[:,:,:3], alpha=0.45)
    axes[2].set_title("Emotion Grad-CAM", fontweight="bold"); axes[2].axis("off")

    asd_scores = {}
    for idx, ((rn, _), cm_) in enumerate(zip(PROP_ROIS.items(), CMAPS)):
        if idx >= 8: break
        sv = sc(asd_cam, masks[rn]); asd_scores[rn] = sv
        heat = asd_cam * (masks[rn].astype(float)/255)
        flag = "[!]" if sv < 0.80 else "[OK]"
        ax = axes[idx+3]
        ax.imshow(img_disp); ax.imshow(cm_(heat)[:,:,:3], alpha=0.55)
        ax.set_title(f"{rn}\n{sv} {flag}", fontweight="bold", fontsize=8); ax.axis("off")

    if "Left Eye" in asd_scores and "Right Eye" in asd_scores:
        l, r = asd_scores["Left Eye"], asd_scores["Right Eye"]
        asd_scores["Eye Sym"] = round(1 - abs(l-r)/(l+r+1e-8), 2)

    ax_bar = axes[11]
    colors = ["#ef5350" if v < 0.80 else "#66bb6a" for v in asd_scores.values()]
    ax_bar.barh(list(asd_scores.keys()), list(asd_scores.values()), color=colors)
    ax_bar.axvline(0.80, color="gray", linestyle="--", lw=1.5, label="Threshold")
    ax_bar.set_xlim(0, 2.5); ax_bar.set_title("ASD Attention Scores", fontweight="bold", fontsize=9)
    ax_bar.legend(fontsize=7)

    ax_rep = axes[12]; ax_rep.axis("off")
    lines = ["FINAL CLINICAL REPORT", "="*26,
             f"ASD Result : {asd_label}",
             f"Emotion    : {emo_label}", ""]
    for k, v in asd_scores.items():
        thr = 0.85 if k == "Eye Sym" else 0.80
        lines.append(f"{k:<14}: {v}  {'[!] Low' if v<thr else '[OK] Normal'}")
    bg = "#fff3e0" if is_asd else "#e8f5e9"
    ax_rep.text(0.03, 0.97, "\n".join(lines), transform=ax_rep.transAxes,
               fontsize=8, va="top", fontfamily="monospace",
               bbox=dict(boxstyle="round", facecolor=bg, alpha=0.9))
    for ax in axes[13:]: ax.axis("off")
    plt.tight_layout()
print("Running clinical XAI on 3 test samples...")
for i in range(min(3,len(imgs))):
    sig=float(asd_model.predict(imgs[i:i+1],verbose=0)[0][0])
    conf=round((sig if sig>0.5 else 1-sig)*100,1)
    asd_lbl=("Non-ASD" if sig>0.5 else "ASD")+f" ({conf}%)"
    ei=min(i,len(emo_imgs)-1)
    er=emo_model.predict(emo_imgs[ei:ei+1],verbose=0)[0]
    en=list(emo_tr.class_indices.keys())
    emo_lbl=en[np.argmax(er)]+f" ({round(float(er.max())*100,1)}%)"
    clinical_xai(asd_model,emo_model,imgs[i:i+1],imgs[i],asd_lbl,emo_lbl)


## STEP 10: Test with Your Own Photo

In [ ]:
from google.colab import files
uploaded=files.upload()
for fname,data in uploaded.items():
    arr=np.frombuffer(data,np.uint8)
    img=cv2.imdecode(arr,cv2.IMREAD_COLOR)
    rgb=cv2.cvtColor(img,cv2.COLOR_BGR2RGB)
    rs=(cv2.resize(rgb,(224,224)).astype(np.float32)/255.0)
    b=np.expand_dims(rs,0)
    sig=float(asd_model.predict(b,verbose=0)[0][0])
    conf=round((sig if sig>0.5 else 1-sig)*100,1)
    asd_r=("Non-ASD" if sig>0.5 else "ASD")+f" ({conf}%)"
    er=emo_model.predict(b,verbose=0)[0]
    en=list(emo_tr.class_indices.keys())
    top3=sorted(zip(en,er.tolist()),key=lambda x:-x[1])[:3]
    emo_r=en[np.argmax(er)]+f" ({round(float(er.max())*100,1)}%)"
    print(f"[1] ASD: {asd_r}")
    print(f"[2] Emotion: {emo_r}  Top-3: {[(e,round(p*100,1)) for e,p in top3]}")
    print("[3] Generating XAI...")
    clinical_xai(asd_model,emo_model,b,rs,asd_r,emo_r)
    print(f"FINAL: {asd_r}  |  {emo_r}")


In [ ]:
from google.colab import files
for f in ["efficientnetv2_asd_model.h5","efficientnetv2_emotion_model.h5",
          "efficientnetv2_asd_metrics.json","efficientnetv2_emotion_metrics.json"]:
    if os.path.exists(f): files.download(f); print(f"Downloaded: {f}")
    else: print(f"Not found: {f}")
